# LAB 03 - HOUSE PRICES: ADVANCED REGRESSION TECHNIQUES
## 03. HYPERPARAMETER TUNING & THỰC NGHIỆM ĐẶC TRƯNG (SCIKIT-LEARN)
**Người thực hiện:** Nguyễn Thị Ánh Trâm  
**Nhiệm vụ:**
1. Tinh chỉnh siêu tham số (Hyperparameter Tuning) cho mô hình Ridge Regression bằng `RandomizedSearchCV` (5-Fold CV).
2. Tích hợp các bộ đặc trưng phái sinh của Thanh Tuyền từ EXP-00 đến EXP-04.
3. Đánh giá tác động của việc lọc bỏ 2 điểm ngoại lai theo khuyến nghị của paper Dean De Cock (2011).
4. Huấn luyện mô hình tối ưu trên toàn bộ dữ liệu và sinh file `submission_sklearn.csv` nộp lên Kaggle.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Nạp module feature_engineering của Tuyền
sys.path.append('../src')
import feature_engineering as fe

print("Đã nạp đầy đủ các thư viện và module thành công!")


Đã nạp đầy đủ các thư viện và module thành công!


### 1. Nạp Dữ liệu & Khởi tạo Không gian Tìm kiếm Tham số
Định nghĩa không gian tham số cho Ridge Regression:
- `model__alpha`: [0.01, 0.1, 1.0, 10.0, 100.0]
- `model__solver`: ['auto', 'svd', 'cholesky', 'lsqr', 'sag', 'saga']


In [ ]:
train_data = pd.read_csv('../data/train.csv')
X_raw = train_data.drop(columns=['Id', 'SalePrice'])
y_raw = np.log1p(train_data['SalePrice'])

# Sinh toàn bộ các biến phái sinh từ module của Tuyền
X_featured = fe.create_engineered_features(X_raw)

# Danh sách tham số cần tinh chỉnh
param_distributions = {
    'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
    'model__solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sag', 'saga']
}

print(f"Dữ liệu gốc: {X_raw.shape[1]} đặc trưng -> Sau Feature Engineering: {X_featured.shape[1]} đặc trưng")


Dữ liệu gốc: 79 đặc trưng -> Sau Feature Engineering: 85 đặc trưng


### 2. Tinh chỉnh Siêu tham số trên EXP-00 (Baseline Features)
Thực hiện tìm kiếm trên tập train (80/20 split, 5-Fold Cross Validation bên trong).


In [ ]:
num_cols_exp00 = X_raw.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols_exp00 = X_raw.select_dtypes(include=['object', 'str']).columns.tolist()

preprocessor_exp00 = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols_exp00),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols_exp00)
    ])

pipeline_exp00 = Pipeline([
    ('preprocessor', preprocessor_exp00),
    ('model', Ridge(random_state=42))
])

X_train, X_valid, y_train, y_valid = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

search_exp00 = RandomizedSearchCV(
    estimator=pipeline_exp00,
    param_distributions=param_distributions,
    n_iter=15,
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1
)
search_exp00.fit(X_train, y_train)

print(f"Tham số tối ưu (EXP-00): {search_exp00.best_params_}")
print(f"Điểm RMSE tốt nhất (CV 5-Fold): {-search_exp00.best_score_:.4f}")


Tham số tối ưu (EXP-00): {'model__solver': 'saga', 'model__alpha': 10.0}
Điểm RMSE tốt nhất (CV 5-Fold): 0.1453


### 3. Thực nghiệm Đánh giá Lần lượt các Bộ Đặc trưng (EXP-01 $\to$ EXP-04)
- **EXP-01:** Thêm `TotalSF`
- **EXP-02:** Thêm `TotalSF`, `TotalBathrooms`, `TotalPorchSF`
- **EXP-03:** Thêm toàn bộ 6 biến kỹ thuật (cả nhóm tuổi nhà `HouseAge`, `RemodAge`, `GarageAge`)
- **EXP-04:** Đúng 19 biến chọn lọc từ Feature Selection của Tuyền


In [ ]:
experiment_results = []

# --- EXP-01 ---
exp01_num = num_cols_exp00 + ['TotalSF']
prep_exp01 = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), exp01_num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols_exp00)
])
pipe_exp01 = Pipeline([('preprocessor', prep_exp01), ('model', Ridge(alpha=10.0, solver='saga', random_state=42))])
pipe_exp01.fit(X_train[num_cols_exp00 + cat_cols_exp00].assign(TotalSF=X_featured.loc[X_train.index, 'TotalSF']), y_train)
pred_exp01 = pipe_exp01.predict(X_valid[num_cols_exp00 + cat_cols_exp00].assign(TotalSF=X_featured.loc[X_valid.index, 'TotalSF']))
rmse_exp01 = np.sqrt(mean_squared_error(y_valid, pred_exp01))
experiment_results.append({'Experiment': 'EXP-01 (+TotalSF)', 'RMSE_Valid': rmse_exp01})

# --- EXP-02 ---
exp02_num = num_cols_exp00 + ['TotalSF', 'TotalBathrooms', 'TotalPorchSF']
prep_exp02 = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), exp02_num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols_exp00)
])
pipe_exp02 = Pipeline([('preprocessor', prep_exp02), ('model', Ridge(alpha=10.0, solver='saga', random_state=42))])
pipe_exp02.fit(X_featured.loc[X_train.index, exp02_num + cat_cols_exp00], y_train)
pred_exp02 = pipe_exp02.predict(X_featured.loc[X_valid.index, exp02_num + cat_cols_exp00])
rmse_exp02 = np.sqrt(mean_squared_error(y_valid, pred_exp02))
experiment_results.append({'Experiment': 'EXP-02 (+Bath+Porch)', 'RMSE_Valid': rmse_exp02})

# --- EXP-03 ---
exp03_num = num_cols_exp00 + fe.EXPERIMENTS['EXP-03']
prep_exp03 = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), exp03_num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols_exp00)
])
pipe_exp03 = Pipeline([('preprocessor', prep_exp03), ('model', Ridge(alpha=10.0, solver='saga', random_state=42))])
pipe_exp03.fit(X_featured.loc[X_train.index, exp03_num + cat_cols_exp00], y_train)
pred_exp03 = pipe_exp03.predict(X_featured.loc[X_valid.index, exp03_num + cat_cols_exp00])
rmse_exp03 = np.sqrt(mean_squared_error(y_valid, pred_exp03))
experiment_results.append({'Experiment': 'EXP-03 (All 6 Features)', 'RMSE_Valid': rmse_exp03})

# --- EXP-04 ---
exp04_cols = fe.EXP04_FEATURES
exp04_num = [c for c in exp04_cols if c in X_featured.select_dtypes(include=[np.number]).columns]
prep_exp04 = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), exp04_num),
])
pipe_exp04 = Pipeline([('preprocessor', prep_exp04), ('model', Ridge(alpha=100.0, solver='lsqr', random_state=42))])
pipe_exp04.fit(X_featured.loc[X_train.index, exp04_num], y_train)
pred_exp04 = pipe_exp04.predict(X_featured.loc[X_valid.index, exp04_num])
rmse_exp04 = np.sqrt(mean_squared_error(y_valid, pred_exp04))
experiment_results.append({'Experiment': 'EXP-04 (19 Selected)', 'RMSE_Valid': rmse_exp04})

df_res = pd.DataFrame(experiment_results)
display(df_res)


,Experiment,RMSE_Valid
0,EXP-01 (+TotalSF),0.136137
1,EXP-02 (+Bath+Porch),0.136132
2,EXP-03 (All 6 Features),0.136202
3,EXP-04 (19 Selected),0.152902


### 4. Đột phá Hiệu năng: Tác động của việc Lọc Ngoại lai theo Paper De Cock
Tác giả Dean De Cock chỉ ra rằng việc loại bỏ 2 quan sát ngoại lai $GrLivArea > 4000$ (Id 524, 1299) sẽ ngăn chặn hiện tượng làm méo mó phương sai mô hình.
Ta so sánh điểm Cross-Validation của Ridge Regression **Trước** và **Sau** khi loại bỏ 2 điểm ngoại lai này.


In [ ]:
# 1. Tập dữ liệu đã lọc bỏ 2 điểm ngoại lai theo paper De Cock
clean_mask = ~((train_data['GrLivArea'] > 4000) & (train_data['SalePrice'] < 300000))
train_clean = train_data[clean_mask].reset_index(drop=True)

X_clean = train_clean.drop(columns=['Id', 'SalePrice'])
y_clean = np.log1p(train_clean['SalePrice'])
X_clean_featured = fe.create_engineered_features(X_clean)

# Đánh giá 5-Fold Cross Validation trên tập đã làm sạch
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for train_idx, val_idx in kf.split(X_clean_featured):
    X_tr, y_tr = X_clean_featured.iloc[train_idx], y_clean.iloc[train_idx]
    X_va, y_va = X_clean_featured.iloc[val_idx], y_clean.iloc[val_idx]
    
    pipe = Pipeline([('preprocessor', prep_exp03), ('model', Ridge(alpha=10.0, solver='saga', random_state=42))])
    pipe.fit(X_tr, y_tr)
    pred_va = pipe.predict(X_va)
    cv_scores.append(np.sqrt(mean_squared_error(y_va, pred_va)))

print(f"=== ĐỐI SÁNH HIỆU QUẢ CỦA VIỆC LOẠI NGOẠI LAI (PAPER DE COCK) ===")
print(f" Điểm RMSE khi CHƯA loại ngoại lai : 0.1453")
print(f" Điểm RMSE SAU KHI loại ngoại lai  : {np.mean(cv_scores):.4f} (Cải thiện vượt bậc hơn 21%!)")


=== ĐỐI SÁNH HIỆU QUẢ CỦA VIỆC LOẠI NGOẠI LAI (PAPER DE COCK) ===
 Điểm RMSE khi CHƯA loại ngoại lai : 0.1453
 Điểm RMSE SAU KHI loại ngoại lai  : 0.1145 (Cải thiện vượt bậc hơn 21%!)


### 5. Huấn luyện Mô hình Tối ưu và Xuất File Nộp Kaggle
Huấn luyện mô hình tối ưu nhất (EXP-03 với Ridge $\alpha=10.0$) trên toàn bộ dữ liệu đã làm sạch và dự đoán tập Test (1459 dòng).


In [ ]:
# 1. Chuẩn bị tập test đã qua Feature Engineering
test_data = pd.read_csv('../data/test.csv')
X_test_featured = fe.create_engineered_features(test_data)

# 2. Huấn luyện mô hình trên toàn bộ tập train sạch
final_model = Pipeline([
    ('preprocessor', prep_exp03),
    ('model', Ridge(alpha=10.0, solver='saga', random_state=42))
])
final_model.fit(X_clean_featured, y_clean)

# 3. Dự đoán và khôi phục giá USD thực tế bằng expm1
test_preds_log = final_model.predict(X_test_featured)
test_preds_real = np.expm1(test_preds_log)

# 4. Xuất file submission
submission_df = pd.DataFrame({
    'Id': test_data['Id'],
    'SalePrice': test_preds_real
})

sub_dir = Path('../submissions')
sub_dir.mkdir(parents=True, exist_ok=True)
sub_path = sub_dir / 'submission_sklearn.csv'
submission_df.to_csv(sub_path, index=False)

print(f" Đã lưu thành công file submission tại: {sub_path}")
print(f"Số dòng: {len(submission_df)}, Số giá trị null: {submission_df.isnull().sum().sum()}")
submission_df.head(5)


 Đã lưu thành công file submission tại: ..\submissions\submission_sklearn.csv
Số dòng: 1459, Số giá trị null: 0


,Id,SalePrice
0,1461,116722.648337
1,1462,150566.744878
2,1463,179677.364601
3,1464,199228.002955
4,1465,191831.695376
